# Laguna, zero to hero: build a coordinated flume experiment

This notebook walks you from "I just cloned the repo" to a **single script-driven
experiment in which every instrument runs on one clock, one schedule, and one
event log.** It is a tutorial, not a reference: it introduces one component at a
time, tests it on its own, then ties them together.

**The experiment we will build** (a stand-in for a bed-evolution run):

| What | Hardware | When |
|---|---|---|
| Water level logged | Massa ultrasonic **gauge** | every 15 s |
| Weir gate and pump follow a time-series | **weir** + **pump** (with inlet/aux valves) | rows of a schedule CSV |
| Topographic transect along a line | **OD2000 rangefinder** carried by the **gantry** | every 200 s |
| Line-laser surface scan | **Gocator** carried by the **gantry** | every 200 s, offset 100 s from the transect |
| Stills of the bed from above | a **pair of Canon DSLRs** | every 100 s |
| Wide-area stills | a **network of three Pi cameras** | every 200 s |

**Sections**

0. Setup and the two safety switches
1. The mental model (five ideas that explain everything else)
2. Configuration: one YAML file describes the rig, one describes the schedule
3. Bench-test each subsystem on its own
4. Safety verbs: pause, resume, stop, estop
5. Design the timeline before you run it
6. Assemble and run the coordinated experiment
7. Afterwards: the event log, the run manifest, and plotting your data
8. Taking it to real hardware, and where to read next

> **You do not need any hardware to run this notebook.** With the defaults below it
> runs in *rehearsal mode* (`simulate=True`): the scheduler, clock, event log, fence
> checks, and safety verbs are all real, and only the wire to the instruments is faked.
> Read the limits of that in section 0.

## 0. Setup and the two safety switches

Two switches at the top of the notebook decide whether anything physical can happen.
**Both default to the safe choice.** Change them yourself, deliberately, and only when
you have read the relevant section.

| `SIMULATE` | `ALLOW_MOTION` | What this notebook does |
|---|---|---|
| `True` | *(ignored)* | **Rehearsal.** No hardware is contacted. Safe anywhere. |
| `False` | `False` | **Real hardware, read-only.** Connects, reads sensors, fires cameras. Sends *no* gantry, weir, pump, or valve commands. |
| `False` | `True` | **Real hardware, live.** The gantry, weir gate, pump and valves can move. Everyone clear, someone on the e-stop. |

`ALLOW_MOTION` is the one switch for *anything that physically moves or changes the
water*: gantry, weir gate, pump, valves. Every cell below that does so is guarded by
`MOTION_OK`, and prints what it skipped when it is `False`.

### What rehearsal mode can and cannot tell you

It **can** catch: a typo in a config key, a subsystem that was never added, a
schedule that fires in the wrong order, two actions that collide, a move that
violates a fence.

It **cannot** catch: a mounting sign error, a feed rate the gantry cannot hold, a
target outside the real work envelope, a valve that does not actually open. Every
sensor reading comes back `NaN` rather than a plausible-looking fake number. It
is a rehearsal of the *script*, not a simulator of the *flume*.

Two more limits you will meet below, both found while writing this tutorial:

* **Rangefinder transects cannot be rehearsed end to end.** The simulated gantry
  connection does not implement the Pi-side scan agent that
  `lab.acquire_scan("od2000", ...)` uses, so in rehearsal we run the transect
  *plan* (`dry_run=True`) and skip the physical pass. Gocator passes do run.
* **`speed_factor` compresses the clock, not the simulated actions.** A simulated
  Gocator pass still takes real wall-clock seconds (its travel time), so at a high
  `SPEED_FACTOR` scheduled actions overlap and the lab pauses itself. That is the
  overlap guard working correctly, not a bug. Keep `SPEED_FACTOR` at about 5 here: the
  first Gocator pass alone (it has to drive Z down to working height) takes ~65
  experiment-seconds at that setting.

In [ ]:
# ---- THE SWITCHES --------------------------------------------------------
SIMULATE = True        # True: rehearse with no hardware.  False: talk to the real rig.
ALLOW_MOTION = False   # Only consulted when SIMULATE is False. See the table above.
SPEED_FACTOR = 5       # Rehearsal only: experiment-seconds per real second. Keep it LOW (see below).

# ---- derived (do not edit) ----------------------------------------------
MOTION_OK = SIMULATE or ALLOW_MOTION
if not SIMULATE:
    SPEED_FACTOR = 1   # real hardware always runs in real time

print(f"mode: {'REHEARSAL (simulate=True)' if SIMULATE else 'REAL HARDWARE'}"
      f"  |  motion/actuation {'ENABLED' if MOTION_OK else 'DISABLED'}"
      f"{'' if not SIMULATE else '  (nothing real can move)'}")

In [ ]:
import logging
import os
import time
from pathlib import Path

import pandas as pd
import yaml

# All files this tutorial writes go under <repo>/tmp/tutorial (tmp/ is git-ignored).
REPO = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "pyproject.toml").exists())
WORKDIR = REPO / "tmp" / "tutorial"
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)   # config paths like ./data/scans resolve here
print("working directory:", Path.cwd())

from laguna import FlumeLab
from laguna.experiment import schedule_action, setup_run
from laguna.survey import SurveyRunner, Traverse

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:
    HAVE_MPL = False
    print("matplotlib not installed: plots are skipped (pip install 'laguna[viz]')")


def console_level(level):
    """Set how chatty laguna is in this notebook (INFO = narrate, WARNING = quiet)."""
    for handler in logging.getLogger().handlers:
        handler.setLevel(level)

console_level(logging.INFO)

## 1. The mental model

Everything in Laguna is built from five ideas. If you hold these, the rest of the
code reads as variations.

```
                       ┌───────────────────── FlumeLab ──────────────────────┐
  config.yaml ───────▶ │  clock · scheduler · event log · run manifest        │
  (what exists,        │  frames (instrument offsets) · safety verbs          │
   when it acts)       └──┬──────┬──────┬──────┬───────┬────────┬────────┬───┘
                          │      │      │      │       │        │        │
                        weir   flow   gauge  gantry  od2000  gocator  cameras
                         (each one is a "subsystem": same shape, same verbs)
```

**1. Opt-in subsystems.** `FlumeLab` knows nothing about weirs or pumps. You attach
what you have with `lab.add(...)`. A config *section present* means "I have this
hardware"; *absent* means it is never constructed.

| Name (`lab.<name>`) | Config section | Hardware | Reached via |
|---|---|---|---|
| `weir` | `weir:` | stepper-driven weir gate (ClearCore) | MQTT, through the confluence node |
| `flow` | `flow:` | Fuji VFD pump + `qin`/`qaux` solenoid valves | MQTT, through the confluence node |
| `gauge` | `gauge:` | Massa ultrasonic water-level sensor | MQTT, through the confluence node |
| `gantry` | `gantry:` | Modusystems OEM-2T, axes X Y Z Theta | SSH to a Pi agent, then RS-232 |
| `od2000` / `wtt12l` | `od2000:` / `wtt12l:` | laser rangefinders (via an ifm AL1342 IO-Link master) | HTTP/MQTT to the AL1342 |
| `gocator` | `gocator:` | LMI Gocator line-laser 3D scanner | Ethernet, GoSdk |
| `dslr_cameras` | `dslr_cameras:` | Canon EOS bodies, any number, bound by serial | gphoto2 over USB |
| `pi_cameras` | `pi_cameras:` | Raspberry Pi camera nodes | SSH |

**2. Every subsystem has the same shape.** `from_config()`, `connect()`,
`disconnect()`, `get_status()`, and the four safety verbs. Learn one, you know them
all. (Adding your own is in `docs/ADDING_A_SUBSYSTEM.md`.)

**3. One clock, two kinds of time.** `lab.clock` tracks *runtime* (seconds of
experiment, which **stops while paused**) alongside *wall time*. Schedules are written
in runtime. The run manifest (section 7) records the mapping so a file timestamp can
always be converted back to runtime.

**4. A scheduler that fires actions on that clock.** `scheduler.repeat(every=...)`
and `scheduler.at(runtime_s)`. Each firing runs in its own thread, so a slow camera
does not delay the pump. You will usually reach it through config keys
(`interval_s`, `trigger_at`, `use_schedule`) rather than by hand.

**5. Two logs with different jobs.** The **event log** (`experiment_events.csv`) is
the terse *archival* record: state changes and milestones, one row each, shipped with
your data. The **operational log** (`laguna.log`, plus the console) is the chatty
debugging narrative. Do not put bench-test noise in the archival log, which is why
we use separate event logs for bench tests and for the real run.

### Priorities, in order

Laguna's design follows this ordering, and it explains behaviours that otherwise look
fussy:

1. **Safety**: all motion is quickly stoppable, and motion is *never* commanded from
   a stop condition or auto-triggered by a health check. A human says "go".
2. **Data preservation**: a missed scan or dropped frame cannot be redone. So a failed
   capture *pauses the whole lab* rather than being skipped, and the event log says
   what was missed.
3. **Graceful shutdown and easy recovery**: when something goes wrong the rig quiesces
   and the log explains why.

## 2. Configuration: what exists, and when it acts

We split configuration into two YAML documents, then merge them into the files the
lab actually reads:

* **`HARDWARE_YAML`**: *what exists* and how to reach it. Changes when you rewire.
* **`SCHEDULE_YAML`**: *when each thing acts*. Changes every experiment.

The split is a convention of this tutorial (laguna itself reads one merged file), but
it is a good habit: scheduling keys (`interval_s`, `trigger_at`, `use_schedule`) are
only consumed by `setup_run()`. They are ignored when you build the lab by hand with
`lab.add()`, which is exactly what we do in the bench-test section.

> **Every value below that names a host, serial number, port, or geometry is a
> placeholder** copied from the repo's `config/example_config.yaml`. In rehearsal they
> are never used. On real hardware you must replace each one. The notebook refuses to
> run with `SIMULATE = False` while any `CHANGEME` remains.

In [ ]:
CALIB = REPO / "config" / "example_pump_calibration.csv"   # replace with YOUR pump's fit

HARDWARE_YAML = f"""
# ---- shared plumbing -----------------------------------------------------
mqtt:
  node_name: "UCRS Confluence Node 1"   # must match "Node Name" in the confluence node's
                                        # config on red.lab. weir, flow and gauge topics
                                        # are derived from it.

# ---- hydraulics (all three are MQTT clients of the confluence node) ------
weir:
  home_offset_mm: 0.0                   # position value assigned at the limit switch on home()
flow:
  calibration_file: {CALIB}             # pump curve: VFD Hz <-> measured L/min. Per-pump!
gauge:
  sensor_index: 0                       # which Massa device ID this gauge reads
  offset_mm: 0.0                        # elevation_mm = offset_mm - measured_distance_mm

# ---- gantry --------------------------------------------------------------
gantry:
  transport: pi_agent                   # SSH to gantry_agent.py on the Pi; the only transport
                                        # that supports rangefinder transects
  host: red.lab
  ssh_user: oak
  ssh_key: ~/.ssh/id_ed25519
  remote_serial_device: /dev/serial/by-id/usb-FTDI_USB-RS232_Cable_AV0K9L0C-if00-port0
  safe_mode: true                       # query-only. KEEP TRUE; section 3.4 explains how motion is enabled
  mm_per_acp_unit: 15.0                 # raw controller unit -> mm. Rig-specific stopgap, see
                                        # docs/archive/GANTRY_UNIT_CALIBRATION.md
  axes:
    - name: X
      index: 1
      home_switch: home
      home_trip_on_high: false
      soft_negative_limit_mm: -5
      soft_positive_limit_mm: 2130
      max_travel_mm: 8000
    - name: Y
      index: 2
      brake_output: 4
      home_switch: home
      home_trip_on_high: false
      soft_negative_limit_mm: -5
      soft_positive_limit_mm: 1200
      max_travel_mm: 1200
    - name: Z
      index: 5
      brake_output: 5
      mm_per_unit: 13.5                 # per-axis override of mm_per_acp_unit
      home_switch: home
      home_trip_on_high: false
      soft_negative_limit_mm: 0
      soft_positive_limit_mm: 520
      max_travel_mm: 400
    - name: Theta
      index: 6
  fences:                               # keep-out zones, checked in pure Python before ANY move
    - type: box
      name: equipment_post
      x: [400, 420]
      y: [400, 420]
      z: [0, 50]

# ---- where each instrument measures, relative to the gantry's commanded point ----
frames:
  instruments:
    gocator:
      translation: [0, 0, -325]         # sensor looks 325 mm below the commanded point
    od2000:
      translation: [52.0, -18.0, 0]     # dot is offset from the commanded point in X and Y
  # experiment:                         # optional: move the origin to a flume corner so all
  #   translation: [500, 300, 0]        # output coordinates are positive

# ---- instruments the gantry carries --------------------------------------
od2000:
  topic: laguna/od2000
  pdin_port: 2                          # IO-Link port on the AL1342 (confirm on your rig)
  offset_mm: 0.0
  al1342_host: 192.168.1.251            # raw IP: needed for laser on/off and transects
  output_dir: ./data/profiles           # where transect CSVs land
  # calibration_file: path/from/scripts/calibrate_rangefinder.py.csv
gocator:
  ip: 192.168.1.10
  travel_speed_mm_s: 20.0               # MUST equal the gantry feed rate used while scanning:
                                        # with no encoder, this is what scales the Y axis
  frame_rate_hz: 140                    # Y spacing = 20 / 140 = 0.14 mm between profiles
  fixed_length_mm: 200.0
  output_dir: ./data/scans
  scan:
    axis: X                             # SurveyRunner overrides axis/end/feed per pass;
    end_mm: 300                         # `formats` is what makes each scan get SAVED
    feed_rate_mm_s: 20
    formats: [npz]
  # mounting:                           # how the sensor's own axes sit on the gantry. RIG-SPECIFIC:
  #   scan_x: +Y                        # see docs/subsystems/scanner.md before enabling.
  #   scan_y: -X
  #   scan_z: +Z

# ---- cameras -------------------------------------------------------------
dslr_cameras:
  capture_target: card                  # keep a backup copy on the SD card (one value for ALL bodies)
  cameras:
    Camera1:
      serial: "CHANGEME-1"              # EOS body serial. List with `python -m laguna.camera.canon`
      imageformat: RAW + L
      exposure: {{iso: "800", aperture: "5.6", shutter: "1/125"}}   # body must be on M, lens on MF
      output_dir: ./captures/Camera1
    Camera2:
      serial: "CHANGEME-2"
      imageformat: RAW + L
      exposure: {{iso: "800", aperture: "5.6", shutter: "1/125"}}
      output_dir: ./captures/Camera2
pi_cameras:
  hosts: [pi1.local, pi2.local, pi3.local]
  ssh_user: ucrs
  ssh_key: ~/.ssh/id_ed25519
  output_dir: ./captures/pi
"""

hardware = yaml.safe_load(HARDWARE_YAML)
print("sections present (each one opts a subsystem in):", ", ".join(hardware))

### When each thing acts

Now the schedule. Three mutually exclusive trigger styles are available to any
scheduled section (laguna raises a `ValueError` at setup time if you set two):

| Key | Meaning |
|---|---|
| `interval_s: N` | fire every N runtime seconds (first firing at runtime N) |
| `trigger_at: [t1, t2, ...]` | fire once at each listed runtime second |
| `use_schedule: true` | fire at every `time_s` row of the schedule CSV (weir/flow use that row's *values*) |

The two gantry-borne instruments (`gocator_transect`, `od2000_transect`) are **not**
built-in scheduled actions. They are *user-defined sections* that we attach our own
actions to in section 6, using the same machinery. The timeline is deliberate:
gantry work and camera exposures are **interleaved, never concurrent**, so the
gantry is parked and still whenever a camera fires.

In [ ]:
SCHEDULE_YAML = """
gauge:
  interval_s: 15            # read every 15 s
  log_as_event: true        # ...and write each reading to the ARCHIVAL event log
                            # (default is operational log only; a level time-series
                            # is data, so we archive it)
weir:
  use_schedule: true        # follow weir_elevation_mm in the schedule CSV
flow:
  use_schedule: true        # follow pump_flow_lpm / qin_open / qaux_open
dslr_cameras:
  trigger_at: [30, 130, 230, 330, 430, 530]   # pair fires together, every 100 s
pi_cameras:
  trigger_at: [20, 220, 420]                  # wide shots, every 200 s
gocator_transect:           # user-defined section: attached to our own action in section 6
  trigger_at: [60, 260, 460]
od2000_transect:
  trigger_at: [160, 360, 560]
"""

SCHEDULE_CSV = """time_s,weir_elevation_mm,pump_flow_lpm,qin_open,qaux_open
0,150,3.0,1,0
150,150,3.0,1,0
300,140,5.0,1,0
450,130,5.0,1,0
600,140,0.0,0,0
"""
Path("schedule.csv").write_text(SCHEDULE_CSV)
print(pd.read_csv("schedule.csv").to_string(index=False))

A `use_schedule` actuator (weir, flow) does not just fire at those times: at each
row it reads the *interpolated target* (`weir_elevation_mm` cubic-spline, `pump_flow_lpm`
linear, valves step-held) and commands it. Pump flow must lie inside the range your
pump calibration covers (the example one tops out at 9.6 L/min); asking for more
raises a `ValueError` rather than silently clamping.

Now merge the two into config files. Note what happens when motion is *not* allowed:
weir and flow fall back to read-only status polling, and the gantry sections are
dropped, so a read-only run cannot command anything physical.

In [ ]:
def deep_update(base, extra):
    for key, value in extra.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


def write_config(path, *, stage):
    """Write the merged config for stage 'bench' (hardware only) or 'experiment'."""
    cfg = yaml.safe_load(HARDWARE_YAML)
    if stage == "bench":
        # Bench-test noise stays out of the archival event log used for the real run.
        cfg["timing"] = {"event_log": "./logs/bench_events.csv"}
    else:
        sched = yaml.safe_load(SCHEDULE_YAML)
        if not MOTION_OK:
            # Read-only: poll status instead of following the schedule; no gantry work.
            sched["weir"] = {"interval_s": 30}
            sched["flow"] = {"interval_s": 30}
            sched.pop("gocator_transect")
            sched.pop("od2000_transect")
        deep_update(cfg, sched)
        cfg["timing"] = {
            "event_log": "./logs/experiment_events.csv",
            "checkpoint_file": "./logs/experiment_checkpoint.json",
            "run_dir": "./runs",   # group every output of a run under runs/<run_id>/
        }
    text = yaml.safe_dump(cfg, sort_keys=False)
    if not SIMULATE and "CHANGEME" in text:
        raise SystemExit("Edit HARDWARE_YAML first: placeholders (CHANGEME) remain and "
                         "SIMULATE is False.")
    Path(path).write_text("# generated by the tutorial notebook\n" + text)
    return Path(path).resolve()


BENCH_CFG = write_config("bench.yaml", stage="bench")
EXP_CFG = write_config("experiment.yaml", stage="experiment")
print("wrote", BENCH_CFG.name, "and", EXP_CFG.name)

## 3. Bench-test each subsystem on its own

Before you coordinate anything, prove each piece works alone. For every subsystem the
pattern is the same:

```python
lab.add("name")            # build it from config and register it as lab.name
lab.name.connect()         # open the connection; returns True/False. CHECK IT.
lab.name.get_status()      # read-only snapshot, always safe
lab.name.<verb>(...)       # then, and only then, an action
```

`connect_all()` does the connect step for everything at once; we do it per subsystem
here so a failure points at exactly one device. We build a **bench lab** whose event
log is separate from the real run's.

In [ ]:
bench = FlumeLab(
    str(BENCH_CFG),
    simulate=SIMULATE,
    debug=False,
)
print("clock running?", bench.clock.is_running, "| runtime:", bench.clock.elapsed())

`FlumeLab(...)` created the timing backbone (clock, scheduler, event log) and loaded
config, but **connected to nothing**. Now we add hardware.

### 3.1 The weir

The weir gate sets the downstream water level. It is a stepper behind a ClearCore,
reached over MQTT, so `connect()` is a broker handshake, not a serial port.

In [ ]:
bench.add("weir")
assert bench.weir.connect(), "weir did not connect: check mqtt.node_name and the broker"
print(bench.weir.get_status())

`go_to_elevation(mm)` returns as soon as the hardware **accepts** the command; it does
not wait for the gate to arrive. Call `wait_for_move()` when you need arrival.
`home()` is different: it physically drives the gate to its limit switch.

> In rehearsal, elevation reads back `NaN`: there is no gate to measure.

In [ ]:
if MOTION_OK:
    print("homing (drives the gate to its limit switch)...")
    print("  homed:", bench.weir.home())
    print("  go_to_elevation(150):", bench.weir.go_to_elevation(150.0))
    bench.weir.wait_for_move(timeout=60)
    print("  elevation now:", bench.weir.get_elevation(), "mm")
else:
    print("SKIPPED (ALLOW_MOTION is False): weir.home() / weir.go_to_elevation()")

### 3.2 The pump and its valves

`flow` owns three things: the VFD that drives the pump, and two solenoid valves,
`qin` (inlet) and `qaux` (auxiliary), exposed as boolean properties.

You ask for **litres per minute**; the pump calibration converts that to a VFD
frequency. `set_flowrate()` only sets the setpoint. `start()` actually runs the pump.
A request the calibration cannot satisfy raises rather than clamping:

In [ ]:
bench.add("flow")
assert bench.flow.connect(), "flow did not connect"
print(bench.flow.get_status())

try:
    bench.flow.set_flowrate(50.0)         # far beyond this pump's calibrated range
except ValueError as exc:
    print("\nrefused, as it should:", exc)

In [ ]:
if MOTION_OK:
    # Make sure the flume is filled and the valves are plumbed the way this assumes.
    bench.flow.qin = True                  # open the inlet valve
    bench.flow.set_flowrate(3.0)           # L/min, inside the calibrated range
    bench.flow.start()
    time.sleep(2.0 if not SIMULATE else 0.1)
    print("running:", bench.flow.get_status())
    bench.flow.stop()                      # clean stop: pump off, setpoint dropped
    bench.flow.qin = False
else:
    print("SKIPPED (ALLOW_MOTION is False): pump start / valve changes")

Note the asymmetry that bites people: `pause()` **stops the pump but remembers the
setpoint** so `resume()` restores it, whereas `stop()` ends the run. Section 4 returns
to this.

### 3.3 The gauge

The gauge converts the Massa's downward distance-to-water into an *elevation*:
`elevation = offset_mm - distance`. Readings arrive asynchronously over MQTT; `read_mm()`
returns the latest published sample (and raises if none has arrived yet).
`read_mm_smoothed()` damps surface ripple.

In [ ]:
bench.add("gauge")
assert bench.gauge.connect(), "gauge did not connect"
time.sleep(0.5)
try:
    print("water level :", bench.gauge.read_mm(), "mm")
    print("smoothed    :", bench.gauge.read_mm_smoothed(), "mm")
except Exception as exc:
    print("no reading yet:", exc)
print(bench.gauge.get_status())

### 3.4 The gantry

The gantry is the one subsystem that can hurt equipment or people, so it has the
most layers of protection. Every `move_to()` passes through, in order:

1. **`safe_mode`**: a query-only allowlist, enforced on the PC *and independently on
   the Pi*. It is `true` in config and stays on until your script turns it off.
2. **The halt latch**: refuses motion after `pause()`/`estop()` until it is cleared.
3. **The motion arbiter**: one lock, so a scheduled scan and a manual move cannot
   command the gantry at once.
4. **Fence check**: your keep-out zones, computed in pure Python against the *whole
   path* from the live position, before a single byte goes out.

`move_to()` returns a handle **immediately** (so you can `pause()` mid-move); call
`.wait()` to block until arrival.

In [ ]:
bench.add("gantry")
assert bench.gantry.connect(), "gantry did not connect"
status = bench.gantry.get_status()
print("safe_mode:", status["safe_mode"], "| positions:", status.get("positions"))

**Fences need no hardware at all**, so check yours every time. This asks whether a
straight-line path from the origin to a point inside `equipment_post` is allowed:

In [ ]:
violations = bench.gantry.checker.check_ribbon((0.0, 0.0, 10.0), (410.0, 410.0, 10.0))
print("rejected, as expected:" if violations else "ERROR: path was accepted!")
for v in violations:
    print("  ", v)

**Referencing.** A move is only meaningful if the controller knows where it is.
After a power cycle you must re-establish position by exactly one of:

| Method | Does it move? | Use when |
|---|---|---|
| `lab.gantry.home().wait()` | **yes**, drives to the home switches | the proper reference |
| `lab.gantry.set_position([x, y, z, theta])` | no | you physically placed it at a known spot |
| `lab.gantry.restore_last_position()` | no | resuming a session after a power cycle; needs `position_checkpoint_file` |

This tutorial assumes the gantry is already referenced and does not choose for you.
In rehearsal, positions simply start at zero.

**Enabling motion** is the explicit, auditable step. It needs a live connection, and
on the Pi transport it relaunches the agent with motion permitted:

In [ ]:
if SIMULATE:
    print("rehearsal: the simulated gantry is already motion-enabled "
          "(it can only move a fake axis).")
elif ALLOW_MOTION:
    # >>> Deliberate, separate step. Be sure the area is clear. <<<
    bench.gantry.set_safe_mode(False)
    print("safe_mode is now OFF: this gantry can move.")
else:
    print("safe_mode stays ON (ALLOW_MOTION is False): any move_to() below would be refused.")

In [ ]:
if MOTION_OK:
    x0 = bench.gantry.get_position()
    print("starting at", x0)
    # Absolute move in the gantry's own coordinates: [X, Y, Z, Theta].
    bench.move_to([40.0, 20.0, 10.0, 0.0], speed=10.0).wait()
    # Per-axis keywords move only what you name; the rest are held.
    bench.move_to(X=60.0, speed=10.0).wait()
    print("now at", bench.gantry.get_position())
else:
    print("SKIPPED (ALLOW_MOTION is False): moves")

`move_to()` speaks **gantry** coordinates: where the carriage is commanded. But a
rangefinder and a Gocator are bolted at different places on the carriage (that is what
the `frames:` section recorded). `place()` speaks **instrument** coordinates: "put *this
instrument's measuring point* at *this experiment-frame point*", and does the
subtraction for you. That is what makes a plan like "scan this line" mean the same
thing no matter which sensor does it.

In [ ]:
point = [120.0, 60.0, 0.0]    # an experiment-frame point
for inst in ("od2000", "gocator"):
    target = bench.frames.gantry_target_for(inst, point)
    print(f"to measure {point} with {inst:8s} command gantry -> {[round(float(v), 1) for v in target]}")

if MOTION_OK:
    bench.place("od2000", point, speed=20.0).wait()
else:
    print("SKIPPED (ALLOW_MOTION is False): place()")

### 3.5 The OD2000 rangefinder, and a topographic transect

The OD2000 reports distance to the bed, one point at a time. A *topographic transect*
is the gantry carrying it along a line while the sample stream is recorded, giving a
bed profile along that line.

Two access paths, both through the AL1342 IO-Link master:

* **Monitoring**: a continuous MQTT stream (`get_distance_mm()`). Slow (~2 Hz) but
  always on.
* **Transects**: the Pi-side agent polls the sensor over HTTP at ~380 Hz while the
  gantry travels. This is what scans use.

In [ ]:
bench.add("od2000")
assert bench.od2000.connect(), "od2000 did not connect (is the AL1342 publishing?)"
time.sleep(1.0)
print("latest distance :", bench.od2000.get_distance_mm(), "mm")
print(bench.od2000.get_status())

The laser emitter is switched explicitly. Always pair `activate()` with a
`deactivate()` in a `try/finally`, so an exception never leaves it on:

In [ ]:
bench.od2000.activate()
try:
    print("single on-demand reading:", bench.od2000.read_mm(), "mm")
finally:
    bench.od2000.deactivate()

Now the transect. Laguna describes any repeated measurement as a **survey plan**: a
pure-data object that holds no hardware and can be printed, costed, and inspected
*before anything moves*. A `Traverse` is "this line, with these instruments, N times".
`SurveyRunner` executes a plan one pass at a time: it positions the *instrument's
measuring point* with `place()`, then scans.

Coordinates are experiment-frame millimetres. We keep the geometry in one place because
the timeline checker in section 5 reuses it.

In [ ]:
# Illustrative geometry: REPLACE with a line that is actually inside your flume.
OD_LINE = dict(start=[100.0, 200.0, 20.0], end=[300.0, 200.0, 20.0])   # experiment-frame mm
OD_SPEED = 10.0       # mm/s while measuring
TRAVEL_SPEED = 40.0   # mm/s while repositioning to the start

od_plan = Traverse(**OD_LINE, instruments=["od2000"], scan_speed=OD_SPEED, travel_speed=TRAVEL_SPEED)
print(od_plan.describe())
print(f"\nestimated measuring time: {od_plan.duration_s():.0f} s")

# dry_run logs every pass's gantry targets WITHOUT moving or measuring.
_ = SurveyRunner(bench, od_plan).run(dry_run=True)

In [ ]:
if SIMULATE:
    print("REHEARSAL LIMIT: the simulated gantry has no Pi scan agent, so a rangefinder "
          "transect cannot run here.\nThe plan above is validated; the physical pass is "
          "verified on real hardware.")
elif MOTION_OK:
    bench.od2000.activate()
    try:
        runner = SurveyRunner(bench, od_plan)
        runner.run(keep_results=True)
        profile = runner.results[0]
        print("saved:", profile.path, "|", profile.metadata.get("samples"), "samples")
    finally:
        bench.od2000.deactivate()
else:
    print("SKIPPED (ALLOW_MOTION is False): transect")

The same pass without the survey wrapper is one call, and is worth knowing for quick
interactive work: it infers the scan axis from the one component that differs, saves a
CSV plus a `_meta.json` sidecar (the sidecar is what lets you orient the data later),
and holds the gantry for the whole operation:

```python
result = lab.acquire_scan(
    "od2000",
    start=[0.0, 50.0, 20.0, 0.0],      # FULL gantry vectors [X, Y, Z, Theta]
    end=[100.0, 50.0, 20.0, 0.0],
    feed_rate_mm_s=10.0,               # deliberately has no default
    output="data/profiles/quick.csv",
)
```

(`WTT12L` PowerProx works identically: swap `"od2000"` for `"wtt12l"`; it has no
programmatic laser switch.)

### 3.6 The Gocator

The Gocator is a line laser: one profile across, many profiles along as the gantry
travels. With no encoder, **Y spacing is `travel_speed_mm_s / frame_rate_hz`**, so the
scan speed must match the config value. `SurveyRunner` passes the `Traverse`'s
`scan_speed` through, and we set both to 20 mm/s.

This sensor is an ordinary Ethernet device on the PC's own network (no Pi, no MQTT),
and needs the vendor SDK built once (`scripts/build_gosdk.sh`). Rehearsal needs neither.

In [ ]:
bench.add("gocator")
assert bench.gocator.connect(), "gocator did not connect (SDK built? sensor on the network?)"
area = bench.gocator.get_active_area()
print(f"active area: width {area['width_mm']:.0f} mm, length {area['length_mm']:.0f} mm")

In [ ]:
GO_LINE = dict(start=[100.0, 400.0, 0.0], end=[200.0, 400.0, 0.0])   # experiment-frame mm
GO_SPEED = 20.0   # mm/s: MUST equal gocator.travel_speed_mm_s

go_plan = Traverse(**GO_LINE, instruments=["gocator"], scan_speed=GO_SPEED, travel_speed=TRAVEL_SPEED)
print(go_plan.describe())

if MOTION_OK:
    runner = SurveyRunner(bench, go_plan)
    runner.run(keep_results=True)
    scan = runner.results[0]
    print("valid points:", scan.valid_count)
else:
    print("SKIPPED (ALLOW_MOTION is False): scan")

### 3.7 The DSLR pair

`dslr_cameras` drives any number of Canon bodies **in parallel on one trigger**. Each
is found by its *body serial* on every connect (never by USB port) and refuses to
connect unless its pre-flight passes. Get serials with `python -m laguna.camera.canon`
and set up stable device names with `scripts/setup_dslr_udev.py` (`docs/CAMERA_USB_SETUP.md`).

Body prerequisites that code cannot fix: mode dial on **M**, lens on **MF**, auto
power-off **disabled**.

`capture_all()` returns one `CaptureRecord` per camera. **A failed capture is never
retried**: a frame missed at time *t* cannot be retaken later and mean the same thing.
During a run any failure pauses the whole lab so a human looks; card copies of a shot
whose download failed are kept, and the event log names them.

In [ ]:
bench.add("dslr_cameras")
assert bench.dslr_cameras.connect(), "a DSLR failed its pre-flight (serial? mode dial? power-off?)"
records = bench.dslr_cameras.capture_all()
for name, rec in records.items():
    print(f"{name}: ok={rec.ok}", [p.name for p in rec.files] if rec.ok else rec.error)

### 3.8 The Pi camera network

`pi_cameras` SSHes to each Pi, deploys a small capture agent, and tells every node to
expose at the **same target instant** (`lead_time` seconds ahead), so shots from
different viewpoints are simultaneous to within clock accuracy. That accuracy depends
on the Pis' clocks agreeing with the PC's, so check it before a run.

In [ ]:
bench.add("pi_cameras")
assert bench.pi_cameras.connect()
if not SIMULATE:
    print("reachable  :", bench.pi_cameras.check_connectivity())
    print("clock skew :", bench.pi_cameras.check_clock_sync(), "(seconds, per host)")
else:
    print("(rehearsal: connectivity and clock-sync checks need real hosts)")

results = bench.pi_cameras.trigger_capture()
print(f"{sum(r.success for r in results)}/{len(results)} cameras captured")
images = bench.pi_cameras.fetch_images(results, Path("captures/pi"))
print({host: Path(p).name for host, p in images.items()})

In [ ]:
report = bench.pi_cameras.report_simultaneity(results)
print("spread (ms):", report["spread_ms"], "->", report["assessment"])

In rehearsal the spread prints `nan` and a "POOR" assessment: the timestamps are
placeholders. On real hardware, treat a poor assessment as "fix NTP on the Pis before the
run", not as something to ignore.

### 3.9 All together, still without a schedule

Every subsystem is now attached, so one call summarises the rig:

In [ ]:
for name, st in bench.get_system_status().items():
    if name == "timing":
        continue
    # pi_cameras opens SSH per capture, so it has no standing connection to report.
    connected = st.get("is_connected", "n/a (SSH per capture)")
    print(f"{name:14s} connected: {connected}")

## 4. Safety verbs: pause, resume, stop, estop

Every subsystem implements the same four verbs with the same meaning, and `FlumeLab`
applies them to all of them at once. Each subsystem is guarded individually, so one that
fails to stop cannot prevent the others from being stopped. An estop works through motion
first (the gantry), then hydraulics (weir, pump), then everything else.

| Verb | Meaning | Reversible? |
|---|---|---|
| `lab.pause()` | cease activity, **hold state**, stay connected. Gantry decelerates, pump stops with its setpoint remembered, acquisition aborts, scheduler halts, clock pauses. | `lab.resume()` |
| `lab.end_run()` | end the run cleanly into a disconnectable state | no: start a new run |
| `lab.estop()` | hardest halt the hardware can reach: gantry abort with brakes on and motors off, pump off, valves closed | `lab.rearm()`, which leaves the gantry in `safe_mode` |
| `lab.rearm()` | clear an estop once the cause is fixed | |

Some verbs return a **note about discarded data** ("scan aborted, partial file kept"),
and `FlumeLab` writes it to the event log, because missing data can invalidate an
experiment as thoroughly as bad data.

> **Naming trap:** on `FlumeLab` itself, `lab.stop()` is an *alias for `pause()`* (it predates the
> vocabulary). The "end the run" verb is `lab.end_run()`. On an individual *subsystem*,
> `stop()` does mean "end cleanly".

Keep `lab.pause()` in a cell **by itself** in any live session: `move_to()` does not
block the kernel, so you can run that cell at any moment to stop the rig.

In [ ]:
# Notes you add land in the event log, anchored to the runtime, and can refer to an earlier event.
note_id = bench.log_note("tutorial: demonstrating pause/resume")

bench.pause(reason="tutorial demo")
print("state after pause :", bench.safety_state.name)

bench.resume()
print("state after resume:", bench.safety_state.name)

**Triggers from outside the process.** A health check, a second script, or a person at a
terminal can demand a pause without touching the notebook, by creating a file. A
background thread polls for `./PAUSE`, `./STOP` and `./ESTOP` every 0.1 s (a thread,
not a signal handler, because signals cannot interrupt a blocking serial read).

In [ ]:
bench.watch_for_safety()          # also follows the pump VFD's hardware e-stop line

Path("PAUSE").touch()             # what you would do from another terminal: `touch PAUSE`
time.sleep(0.6)
print("state after touching PAUSE:", bench.safety_state.name)

bench.resume()                    # refused while the trigger is still asserted
print("resume while file exists  :", bench.safety_state.name)

Path("PAUSE").unlink()            # clear the cause...
time.sleep(0.4)
bench.resume()                    # ...then resume
print("after removing file       :", bench.safety_state.name)

`resume()` deliberately **refuses while the trigger is still asserted**: resuming into
the condition that tripped a pause is how a second incident happens.

The hard version is worth rehearsing once so it is never a surprise. It is only
exercised in rehearsal here; practise the real thing with the guided hardware checks in
`examples/example_15_motion_safety_hardware_check.ipynb`.

In [ ]:
if SIMULATE:
    bench.estop(reason="tutorial drill")
    print("after estop:", bench.safety_state.name)
    try:
        bench.resume()
    except RuntimeError as exc:
        print("resume refused:", exc)
    print("rearm ->", bench.rearm(), "| state:", bench.safety_state.name,
          "| gantry safe_mode:", bench.gantry.get_status()["safe_mode"])
else:
    print("Not run on real hardware from this notebook.")

Note the last value: after `rearm()` the gantry is back in **`safe_mode`**. Motion must
be re-enabled explicitly. An estop never silently returns you to a state where the next
`move_to()` just goes.

Bench testing is finished. Stop its sentinel watcher and disconnect it all before the real run takes over the same
devices (the gantry agent allows one connection):

In [ ]:
bench.safety_monitor.stop()    # or its sentinel watcher would also react to the real run's PAUSE/ESTOP files
bench.disconnect_all()
print("bench lab disconnected")

## 5. Design the timeline before you run it

A scheduler will happily fire two things at once. Nothing in the built-in camera actions
knows the gantry is in the field of view or shaking the bed. **Preventing that is your
job**, at design time, and it costs nothing to check.

Rules this experiment follows:

* Gantry actions (transect, scan) never overlap each other. Laguna *enforces* this with
  a motion arbiter and the overlap guard (an overrun escalates to a pause), but a
  design that relies on that guard is a bad design.
* Camera exposures fire only while the gantry is parked.

We estimate each gantry action's duration as a repositioning allowance plus
`length / speed`, then check every trigger against those windows.

In [ ]:
sched = yaml.safe_load(SCHEDULE_YAML)

def line_length(line):
    return sum((e - s) ** 2 for s, e in zip(line["start"], line["end"])) ** 0.5

REPOSITION_ALLOWANCE_S = 30   # generous time to drive to the start of a line
MARGIN_S = 5

gantry_jobs = {
    "gocator_transect": REPOSITION_ALLOWANCE_S + line_length(GO_LINE) / GO_SPEED,
    "od2000_transect":  REPOSITION_ALLOWANCE_S + line_length(OD_LINE) / OD_SPEED,
}
windows = sorted(
    (t, t + gantry_jobs[name], name)
    for name in gantry_jobs for t in sched[name]["trigger_at"]
)

problems = []
for (a0, a1, an), (b0, b1, bn) in zip(windows, windows[1:]):
    if b0 < a1 + MARGIN_S:
        problems.append(f"{an}@{a0} (ends ~{a1:.0f}s) collides with {bn}@{b0}")
for cam in ("dslr_cameras", "pi_cameras"):
    for t in sched[cam]["trigger_at"]:
        for w0, w1, wn in windows:
            if w0 - MARGIN_S <= t <= w1 + MARGIN_S:
                problems.append(f"{cam}@{t} fires while the gantry is busy ({wn} {w0}-{w1:.0f}s)")

DURATION = max(w1 for _, w1, _ in windows) + 30
print("gantry windows (s):", [(t0, round(t1)) for t0, t1, _ in windows])
print("problems:", problems or "none")
assert not problems, "fix the timeline before running"
print(f"\nplanned run length: {DURATION:.0f} s of runtime "
      f"(~{DURATION / SPEED_FACTOR:.0f} s wall clock at SPEED_FACTOR={SPEED_FACTOR})")

In [ ]:
if HAVE_MPL:
    fig, ax = plt.subplots(figsize=(10, 3.4))
    lanes = ["gauge", "weir / pump", "dslr_cameras", "pi_cameras", "gocator", "od2000"]
    y = {name: i for i, name in enumerate(lanes)}
    csv = pd.read_csv("schedule.csv")
    ax.scatter(range(15, int(DURATION), sched["gauge"]["interval_s"]),
               [y["gauge"]] * len(range(15, int(DURATION), sched["gauge"]["interval_s"])), marker="|", s=60, color="#0072B2")
    ax.scatter(csv.time_s, [y["weir / pump"]] * len(csv), marker="D", s=40, color="#009E73")
    for lane, key, col in (("dslr_cameras", "dslr_cameras", "#D55E00"), ("pi_cameras", "pi_cameras", "#E69F00")):
        ax.scatter(sched[key]["trigger_at"], [y[lane]] * len(sched[key]["trigger_at"]), marker="o", s=40, color=col)
    for t0, t1, name in windows:
        lane = "gocator" if name.startswith("gocator") else "od2000"
        ax.barh(y[lane], t1 - t0, left=t0, height=0.5, color="#CC79A7" if lane == "gocator" else "#56B4E9")
    ax.set_yticks(range(len(lanes)), lanes)
    ax.invert_yaxis()
    ax.set_xlim(0, DURATION)
    ax.set_xlabel("experiment runtime (s)")
    ax.set_title("Planned timeline: bars are estimated gantry-busy windows")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    plt.show()

## 6. Assemble and run the coordinated experiment

Everything so far was bottom-up. For the run we switch to **config-driven assembly**.
`setup_run()` is the standard entry point (it is what the repo's experiment scripts
call). It:

1. builds a `FlumeLab` from the config file,
2. calls `add_all()`: every subsystem whose section is present gets built and registered,
3. connects them all (a failure is logged and the run continues without that device),
4. registers one scheduled action per subsystem from its `interval_s` / `trigger_at` /
   `use_schedule` key (weir and flow read their setpoints from the schedule CSV;
   cameras trigger exactly as configured; each action also guards against overlapping
   *itself*).

What it does **not** do is schedule the gantry-borne transects. Those are ours.

In [ ]:
if SIMULATE:
    # Simulation artifact: the model gantry reports zero acceleration; harmless noise.
    logging.getLogger("laguna.robot.macron.gcode").setLevel(logging.ERROR)

lab = setup_run(
    str(EXP_CFG),
    schedule="schedule.csv",
    simulate=SIMULATE,
    speed_factor=SPEED_FACTOR if SIMULATE else None,
)
console_level(logging.WARNING)   # the run is long; show only warnings and errors
print("subsystems:", ", ".join(lab._subsystems))
print("scheduled actions registered so far:",
      len(lab.scheduler._recurring), "recurring +", len(lab.scheduler._oneshot), "one-shot")

### The gantry actions

A scheduled action is a **zero-argument function**. Ours reuse the `Traverse` plans from
section 3, with three habits worth copying:

* **Each firing starts by placing the instrument explicitly.** An action must never
  depend on where the *previous* action happened to leave the gantry.
* **A failure escalates; it never disappears.** `lab.escalate()` pauses the whole lab and
  writes why. Swallowing the exception would leave the schedule marching on, silently
  not collecting the topography it exists to collect. (The same philosophy as DSLR
  failures.)
* **Laser on only for the pass**, in `try/finally`.

`exclusive_with="gantry_busy"` puts both actions in one exclusion group: if one fires
while the other is still running, the lab pauses instead of racing for the gantry.

> Recording the transect file in the run manifest (`lab.run.record_output`) is done
> for the rangefinder below, where laguna returns the file path. Gocator scans land in
> the run's `gocator/` directory regardless (`setup_run` routes `output_dir` there).

In [ ]:
def gocator_transect():
    try:
        SurveyRunner(
            lab,
            Traverse(**GO_LINE, instruments=["gocator"], scan_speed=GO_SPEED, travel_speed=TRAVEL_SPEED),
        ).run()
    except Exception as exc:
        lab.escalate(f"gocator transect failed: {exc}")


def od2000_transect():
    runtime_s = lab.clock.elapsed()
    try:
        plan = Traverse(**OD_LINE, instruments=["od2000"], scan_speed=OD_SPEED, travel_speed=TRAVEL_SPEED)
        if lab.simulate:
            # Rehearsal limit (section 0): validate the plan, skip the physical pass.
            SurveyRunner(lab, plan).run(dry_run=True)
            lab.log_note("[rehearsal] od2000 transect planned but not executed "
                         "(simulated gantry has no scan agent)")
            return
        lab.od2000.activate()
        try:
            runner = SurveyRunner(lab, plan)
            runner.run(keep_results=True)
        finally:
            lab.od2000.deactivate()
        lab.run.record_output("od2000", runner.results[0].path, runtime_s)
    except Exception as exc:
        lab.escalate(f"od2000 transect failed: {exc}")


if MOTION_OK:
    schedule_action(lab, lab.config.get("gocator_transect"), subsystem="gocator",
                    name="transect", action=gocator_transect, exclusive_with="gantry_busy")
    schedule_action(lab, lab.config.get("od2000_transect"), subsystem="od2000",
                    name="transect", action=od2000_transect, exclusive_with="gantry_busy")
    print("gantry transects registered")
else:
    print("ALLOW_MOTION is False: gantry transects NOT registered")
print("scheduled:", len(lab.scheduler._recurring), "recurring +", len(lab.scheduler._oneshot), "one-shot")

### Pre-flight

Do not start a run you have not looked at. The three checks that catch most mistakes:
every subsystem connected, the safety triggers armed, and (on real hardware) motion
enabled only now, as a conscious act.

`watch_for_safety()` arms the sentinel files from section 4. From a terminal in the run
directory, `touch PAUSE` (resumable), `touch STOP`, or `touch ESTOP` now halts the run
within 0.1 s, **and keep a hand near the physical e-stop regardless**.

In [ ]:
status = lab.get_system_status()
unready = [n for n, s in status.items() if n != "timing" and s.get("is_connected") is False]
if not SIMULATE:
    unready += [f"pi_cameras:{h}" for h, ok in lab.pi_cameras.check_connectivity().items() if not ok]
print("not connected:", unready or "none")
assert not unready, "a subsystem failed to connect: see the warnings above; fix it before running"

lab.watch_for_safety()

if not SIMULATE and ALLOW_MOTION:
    lab.gantry.set_safe_mode(False)   # the deliberate step; see section 3.4
    print("gantry safe_mode OFF")
print("safety state:", lab.safety_state.name, "| run id:", lab.run.run_id)

### Go

`lab.start(duration)` starts the clock and scheduler on a **background thread** and
returns immediately: the notebook stays responsive, and `lab.pause()` from any other
cell halts everything. A watch loop below prints progress.

A **pause drill** is included by default: at the first moment after 140 s of runtime when the
gantry is idle, we pause the lab, wait, and resume. (We wait for idle on purpose: pausing
*mid-pass* aborts that pass, the event log records the abort, and the pass is **not** redone
for you. A pause costs you the scan that was in flight.) This exercises the path you most need to trust (the schedule
restarts for the *remaining* time, runtime excludes the pause, and the manifest records
the gap) and gives section 7 something interesting to show. Set `PAUSE_DRILL = False`
to skip it.

On real hardware with motion enabled, the cell asks you to type `GO` first. The
`start()` call is the human "go", and nothing in the schedule will move the rig before it.

In [ ]:
PAUSE_DRILL = True

if not SIMULATE and ALLOW_MOTION:
    if input("Motion is ENABLED. Area clear and e-stop within reach? Type GO to start: ").strip() != "GO":
        raise SystemExit("not started")

thread = lab.start(DURATION)
drilled = False
last_print = -1.0

while True:
    thread.join(timeout=0.5)
    now = lab.clock.elapsed()

    if thread.is_alive():
        if now - last_print >= 60:
            print(f"  t = {now:5.0f} s | state = {lab.safety_state.name}")
            last_print = now
        if PAUSE_DRILL and not drilled and now >= 140 and not lab.gantry.arbiter.is_held:
            drilled = True
            print(f"  >>> pause drill at t = {now:.0f} s")
            lab.pause(reason="tutorial pause drill")
            time.sleep(2.0)                         # the clock is stopped; runtime does not advance
            thread = lab.resume() or thread         # restarts the scheduler for the REMAINING time
            print(f"  >>> resumed, state = {lab.safety_state.name}")
        continue

    if lab.safety_state.name != "RUNNING":          # something escalated a pause: do not plough on
        print(f"stopped early: state = {lab.safety_state.name}: read the log above")
    break

print("done at runtime", round(lab.clock.elapsed()), "s")

If the run **did** stop early, `lab.safety_state` says so, and the reason is the last
`ERROR` line plus a row in the event log (section 7 shows how to read it). Fix the cause,
then `lab.resume()`; do not just re-run the cell.

Then a clean shutdown, which closes the run manifest and leaves no connection (or laser)
behind. (Laguna prints the end-of-run summary by itself only when a run finishes without ever
having been paused and resumed, so the cell prints it for you after a pause drill.)

In [ ]:
if drilled:                      # a natural, never-paused finish already printed it
    lab.print_summary()
lab.disconnect_all()
print("disconnected. run id:", lab.run.run_id)

## 7. Afterwards: the event log, the run manifest, and your data

Three artifacts describe a run, and together they let you reconstruct what happened
and prove nothing was lost.

| Artifact | Where | Answers |
|---|---|---|
| **Event log** | `logs/experiment_events.csv` (`..._simulated.csv` in rehearsal) | What happened, in order, and did it succeed? |
| **Run manifest** | `runs/<run_id>/run.json` | When were we paused, and where are the output files? |
| **Operational log** | `runs/<run_id>/laguna.log` | The detailed narrative, for debugging |

### The event log

In [ ]:
ev_path = Path(lab.config.get_value("timing.event_log"))
if SIMULATE:                      # rehearsals are never allowed to share a file with real data
    ev_path = ev_path.with_name(f"{ev_path.stem}_simulated{ev_path.suffix}")

events = pd.read_csv(ev_path)
start_row = events.index[events.event_type == "experiment_start"][-1]      # only the latest run
events = events.loc[start_row:].reset_index(drop=True)

print(f"{len(events)} events in this run\n")
summary = (events[~events.subsystem.isin(["scheduler"])]
           .groupby(["subsystem", "event_type"]).size().rename("count"))
print(summary.to_string())

**Did anything go wrong?** These are the rows to look at. Failures are written as
`error: ...` in the `result` column; scheduler falls-behind events are written as
`skipped: ...`; pauses and their reasons are rows too. Because data is perishable,
this is the first thing to read after every run:

In [ ]:
res = events.result.fillna("").astype(str)
trouble = events[res.str.startswith(("error", "skipped", "failed"))
                 | events.event_type.isin(["experiment_pause", "experiment_resume", "estop", "rearm"])]
print(trouble[["runtime_s", "subsystem", "event_type", "result", "notes"]]
      .to_string(index=False, max_colwidth=60) if len(trouble) else "no errors, skips or pauses")

With the pause drill on you will see an `experiment_pause` and `experiment_resume` pair
with the reason attached. A real incident looks the same, except the reason says what
tripped it: that is the "logging should explain what led to the pause" requirement
in action.

**Reading the gauge as a time series.** With `log_as_event: true`, each reading is a row
whose `notes` carries `elevation_mm=...`. (In rehearsal these are `nan`, by design.)

In [ ]:
gauge = events[(events.subsystem == "gauge") & (events.event_type == "read_mm")].copy()
gauge["elevation_mm"] = gauge.notes.str.extract(r"elevation_mm=([-\d.naNa]+)")[0].astype(float)
print(gauge[["runtime_s", "elevation_mm"]].head(8).to_string(index=False))

if HAVE_MPL and gauge.elevation_mm.notna().any():
    schedule_df = pd.read_csv("schedule.csv")
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(gauge.runtime_s, gauge.elevation_mm, "-o", ms=3, label="gauge (measured)")
    ax.step(schedule_df.time_s, schedule_df.weir_elevation_mm, where="post", label="weir setpoint")
    ax.set_xlabel("runtime (s)"); ax.set_ylabel("elevation (mm)"); ax.legend(); plt.show()
elif not SIMULATE:
    print("no valid gauge readings: check the gauge topic and offset_mm")
else:
    print("(rehearsal: all readings are NaN by design, so there is nothing to plot)")

### The run manifest: runtime ↔ wall time

A file's timestamp is *wall* time, but your schedule is in *runtime*, and every pause
makes them drift apart. The manifest records each pause interval so you can always
convert, in either direction, even in a fresh Python session next month:

In [ ]:
from laguna.run_context import RunContext

manifest = RunContext.load(str(lab.run.directory / "run.json"))
print("run id  :", manifest.run_id)
print("outputs :", len(manifest.outputs), "files recorded")

# A pause is zero seconds of RUNTIME but real seconds of WALL clock. (The clock also
# stops itself when the schedule finishes, so skip the entry at the very end.)
for paused_at, resumed_at in manifest.pauses:
    if manifest.runtime_at(paused_at) < DURATION - 1:
        print(f"pause   : at runtime {manifest.runtime_at(paused_at):.0f} s, "
              f"lasting {resumed_at - paused_at:.1f} s of wall clock")

# Convert in either direction. Here: when did a scan file get written, in experiment time?
scan_files = sorted(lab.run.directory.glob("gocator/*.npz"))
for f in scan_files:
    print(f"{f.name} was written at runtime {manifest.runtime_at(f.stat().st_mtime):.0f} s")
print(f"runtime 300 s happened at wall time "
      f"{pd.Timestamp(manifest.wall_at(300.0), unit='s', tz='UTC'):%Y-%m-%d %H:%M:%S} UTC")

### Plotting a Gocator scan

Raw scans are in the **sensor's** coordinates. `orient_scan()` moves them into the
experiment frame, using the mounting and the gantry position saved in the scan's own
metadata, so data from different instruments and different days line up. Plotting
before orienting still draws, but landmarks will not match.

This works on rehearsal output too: the simulated Gocator emits a small, fixed, synthetic
surface (a sloped plane with a hole in it; it says nothing about your flume), so you can
check that your plotting pipeline works before you have any real data.

In [ ]:
from laguna.config import Config
from laguna.frames import FrameRegistry, orient_scan
from laguna.scanner.pointcloud import SurfaceScan

scan_files = sorted(lab.run.directory.glob("gocator/*.npz"))
print(f"{len(scan_files)} Gocator scans in this run")

if scan_files and HAVE_MPL:
    from laguna.viz import plot_acquisition

    frames = FrameRegistry.from_config(Config(str(EXP_CFG)).config_dict.get("frames"))
    oriented = orient_scan(SurfaceScan.from_npz(str(scan_files[-1])), instrument="gocator", frames=frames)
    fig, axes = plot_acquisition(oriented, title=scan_files[-1].stem)
    plt.show()
elif not scan_files:
    print("no scans (ALLOW_MOTION is False, or the run stopped before the first one)")

### Plotting a rangefinder transect

A transect CSV has a `_meta.json` sidecar beside it carrying the gantry axis and start
point; load it, and orient the profile the same way. Needs a real transect, so this
cell is a no-op in rehearsal:

```python
import json
from laguna.robot.macron.profiler import ProfileResult, orient_profile
from laguna.viz import plot_acquisition

csv_path = next(Path("data/profiles").glob("*.csv"))
meta = json.loads(csv_path.with_name(csv_path.stem + "_meta.json").read_text())
oriented = orient_profile(ProfileResult(path=csv_path, metadata=meta),
                          instrument="od2000", frames=frames,
                          config=lab.config.get("od2000"))   # pass a calibration for real heights
plot_acquisition(oriented, title=csv_path.stem)
```

Without a `calibration_file`, `orient_profile()` warns and the Z column reflects only
the mount offset, not a real bed height. Fit one with `scripts/calibrate_rangefinder.py`.

### If something *was* missed

Data preservation is the second priority, so recovery is designed in:

* **Find it.** The event log rows above name every failure and every skipped firing, with
  the runtime.
* **Re-acquire it.** For a multi-pass survey, give `SurveyRunner` a `CheckpointStore` and
  it resumes after the last completed pass instead of restarting. For time-triggered
  actions, the runtime in the log tells you which firing to redo by hand. See
  [`docs/guides/scheduled-experiment.md`](../docs/guides/scheduled-experiment.md).
* **Resume.** After a pause, `lab.resume()` continues for the *remaining* time. After a
  crash, a new process with `CheckpointStore(resume=True)` picks up where the old one
  stopped. After a power cycle, `gantry.restore_last_position()`.

## 8. Taking it to real hardware, and where to read next

Do this **in order**. Each step is cheap and each step removes a class of surprise. Do
not skip ahead to motion.

| # | Step | How you know it worked |
|---|---|---|
| 1 | **Rehearse the exact script.** Run this notebook (or your script) with `SIMULATE = True` end to end. | no early pause; event log shows every action once |
| 2 | **Replace every placeholder** in `HARDWARE_YAML`: hosts, serials, ports, IPs, geometry, fences, soft limits. | the notebook stops complaining about `CHANGEME` |
| 3 | **One-time hardware bring-up**: confluence node + `mqtt.node_name`, AL1342 (`docs/MQTT_AL1342_SETUP.md`), WTT12L bridge, DSLR udev rules (`docs/CAMERA_USB_SETUP.md`), Gocator SDK (`scripts/build_gosdk.sh`). | each `connect()` returns `True` |
| 4 | **Calibrate**: pump curve (`docs/subsystems/flow.md`, `src/laguna/flow/calibration.py`), rangefinder (`scripts/calibrate_rangefinder.py`), gantry `mm_per_acp_unit`, Gocator `mounting`. | scan of a known object looks right-handed, not mirrored |
| 5 | **Fix the fences and soft limits** to the real envelope. | `check_ribbon()` rejects a path into every obstacle |
| 6 | **Pi clocks synced** (`check_clock_sync`). DSLR bodies on M / MF / no auto-off. | simultaneity spread assessed GOOD |
| 7 | **Read-only live run.** `SIMULATE = False`, `ALLOW_MOTION = False`. Sections 3 and 6 will connect, read, and fire cameras, but command nothing. | gauge and rangefinder show real numbers |
| 8 | **Practise the stops** with the people who will be present: `touch PAUSE`, `touch ESTOP`, the physical e-stop. Use `example_15`. | everyone can name which stops what |
| 9 | **Reference the gantry** (home, or declare, or restore). | `get_status()` positions match the tape measure |
| 10 | **Shrink the first live run**: short duration, slow speeds, one transect, someone watching. Then flip `ALLOW_MOTION = True`. | you watched it do exactly what the plan said |

### Reading list

| To learn about | Read |
|---|---|
| The architecture in one place | `docs/ARCHITECTURE.md` |
| Rehearsal details | `docs/guides/simulate.md`, `src/laguna/simulation.py` |
| Gantry safety layers | `docs/guides/gantry-motion.md`, `docs/MOTION_CONTROL_LAYERS.md` |
| Pause / stop / estop semantics | `src/laguna/safety.py` module docstring |
| Scheduling and user-defined actions | `docs/subsystems/experiment.md`, `docs/guides/scheduled-experiment.md` |
| Frames and instrument offsets | `docs/subsystems/frames.md` |
| Surveys (`Tile`, `Traverse`) | `src/laguna/survey.py` module docstring, `examples/example_14_*.py` |
| Plotting saved data | `docs/guides/plot-saved-acquisitions.md` |
| Writing your own subsystem | `docs/ADDING_A_SUBSYSTEM.md`, `docs/CONTRIBUTING.md` |
| Every config key | `config/example_config.yaml` (heavily commented) |

### Cheat sheet

```python
lab = FlumeLab("config.yaml", simulate=True)   # or setup_run(...) for config-driven assembly
lab.add("weir").add("gantry")                  # build from config + register
lab.connect_all()                              # CHECK THE RETURN VALUE

lab.move_to([x, y, z, th], speed=10).wait()    # gantry coords; returns a handle immediately
lab.place("od2000", [x, y, z]).wait()          # instrument coords (offset-aware)
SurveyRunner(lab, Traverse(...)).run()         # repeatable, plannable measurement

lab.start(seconds)                             # background run; lab.clock.elapsed() is runtime
lab.pause() / lab.resume()                     # reversible, holds state
lab.estop() / lab.rearm()                      # hardest halt / explicit recovery
lab.log_note("why I did that")                 # operator context into the archival log
lab.disconnect_all()                           # always
```